Spatial Joins for cleaned data

In [ ]:
import pandas as pd
import geopandas as gpd

# Load cleaned data
trees = gpd.read_file("../data/processed/trees_cleaned.geojson")
sensor_daily = pd.read_csv("../data/processed/sensors_daily.csv")
soil_daily = pd.read_csv("../data/processed/soil_daily.csv")
weather = pd.read_csv("../data/processed/weather_cleaned.csv")

# Reproject trees to EPSG:7844
trees = trees.to_crs("EPSG:7844")

# Get unique sensor locations (one row per sensor not per day)
sensor_locations = sensor_daily.drop_duplicates(subset=['sensor_location'])[['sensor_location', 'lat', 'lon']]
sensor_locations_gdf = gpd.GeoDataFrame(
    sensor_locations,
    geometry=gpd.points_from_xy(sensor_locations['lon'], sensor_locations['lat']),
    crs="EPSG:4326"
).to_crs("EPSG:7844")

# Get unique soil sensor locations
soil_locations = soil_daily.drop_duplicates(subset=['site_id'])[['site_id', 'Latitude', 'Longitude']]
soil_locations_gdf = gpd.GeoDataFrame(
    soil_locations,
    geometry=gpd.points_from_xy(soil_locations['Longitude'], soil_locations['Latitude']),
    crs="EPSG:4326"
).to_crs("EPSG:7844")

print(f"Trees: {trees.shape[0]}")
print(f"Unique microclimate sensors: {sensor_locations_gdf.shape[0]}")
print(f"Unique soil sensors: {soil_locations_gdf.shape[0]}")

Trees: 82064
Unique microclimate sensors: 11
Unique soil sensors: 69


In [2]:
# Join each tree to its nearest microclimate sensor
trees_with_sensors = gpd.sjoin_nearest(
    trees,
    sensor_locations_gdf[['sensor_location', 'geometry']],
    how='left',
    distance_col='sensor_distance_m'
)

print(f"Trees with nearest microclimate sensor: {trees_with_sensors.shape}")
print(f"\nDistance to nearest sensor (metres):")
print(trees_with_sensors['sensor_distance_m'].describe())

# Join each tree to its nearest soil sensor
trees_with_sensors = gpd.sjoin_nearest(
    trees_with_sensors.drop(columns=['index_right']),
    soil_locations_gdf[['site_id', 'geometry']],
    how='left',
    distance_col='soil_sensor_distance_m'
)

print(f"\nTrees with both sensors: {trees_with_sensors.shape}")
print(f"\nDistance to nearest soil sensor (metres):")
print(trees_with_sensors['soil_sensor_distance_m'].describe())

/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/Users/aidanpage/Documents/GitHub/MOP-Code/Playground/AidanPage_T126/venv/lib/python3.9/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Trees with nearest microclimate sensor: (82064, 21)

Distance to nearest sensor (metres):
count    82064.000000
mean         0.014189
std          0.010299
min          0.000022
25%          0.006639
50%          0.011951
75%          0.017868
max          0.052376
Name: sensor_distance_m, dtype: float64

Trees with both sensors: (82064, 23)

Distance to nearest soil sensor (metres):
count    82064.000000
mean         0.006912
std          0.007596
min          0.000003
25%          0.001781
50%          0.003806
75%          0.008004
max          0.041724
Name: soil_sensor_distance_m, dtype: float64


In [3]:
# Check columns 
print("Columns:")
print(trees_with_sensors.columns.tolist())

# Save the joined tree data
trees_with_sensors.to_file("../data/processed/trees_with_sensors.geojson", driver="GeoJSON")
print(f"\nSaved: {trees_with_sensors.shape}")

Columns:
['com_id', 'common_name', 'scientific_name', 'genus', 'family', 'diameter_breast_height', 'year_planted', 'date_planted', 'age_description', 'useful_life_expectency', 'useful_life_expectency_value', 'precinct', 'located_in', 'latitude', 'longitude', 'risk_class', 'tree_age', 'geometry', 'sensor_location', 'sensor_distance_m', 'index_right', 'site_id', 'soil_sensor_distance_m']

Saved: (82064, 23)
